In [ ]:
#Train / Validation / Test
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def make_student_success(n=600, seed=1):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })
    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)
    y = np.where(rng.random(n) < 0.07, 1-y, y)
    return X, y

X, y = make_student_success()

# Split 1: Train vs Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=3, stratify=y
)

# Split 2: Validation vs Test (from temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=3, stratify=y_temp
)
# Final ratios: 70% train, 15% val, 15% test

clf = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

val_acc = accuracy_score(y_val, clf.predict(X_val))
test_acc = accuracy_score(y_test, clf.predict(X_test))

print("Validation accuracy:", round(val_acc, 3))
print("Test accuracy:", round(test_acc, 3))

# Validation Set — Explained for Students (Age 16 Friendly)

## Why We Need Three Data Splits

When we build a machine learning model, we usually split the data into **three parts**:

* **Training Data**
* **Validation Data**
* **Test Data**

Each part has a **different job**.

---

# 1. Training Data (Where the Model Learns)

The **training set** is where the model learns patterns.

Example features:

* hours studied
* attendance
* sleep
* practice tests

The model studies these and learns patterns like:

> "Students with these habits usually pass."

This learning happens with:

```python
clf.fit(X_train, y_train)
```

Think of this like **studying for an exam**.

---

# 2. Validation Data (Practice Exam)

The **validation set** is used to check how well the model performs on **new data it hasn't seen before**.

Example code:

```python
val_acc = accuracy_score(y_val, clf.predict(X_val))
```

Here the model makes predictions on the **validation set**, and we measure how accurate it is.

Important rule:

**The model never saw this data during training.**

So this acts like a **practice exam**.

---

# 3. Why Validation Exists

Imagine preparing for a final exam.

Bad strategy:

```
Study → Take final exam → Study → Take final exam
```

After the first time, you would start **memorizing the exam itself**, which is not real learning.

Better strategy:

```
Study → Practice test → Improve → Practice test → Improve → Final exam
```

Machine learning works the same way.

| Stage      | Human Example |
| ---------- | ------------- |
| Training   | Studying      |
| Validation | Practice Exam |
| Testing    | Final Exam    |

---

# 4. What Validation Helps Us Decide

The validation set helps us choose things like:

* Which model works better
* Which parameters work best
* Whether the model is **overfitting**

Example:

```
Model A validation accuracy = 0.82  
Model B validation accuracy = 0.88
```

We choose **Model B** because it performs better on validation data.

---

# 5. Test Set (The Final Exam)

After we finish improving the model using validation data, we evaluate the **final performance** using the **test set**.

```python
test_acc = accuracy_score(y_test, clf.predict(X_test))
```

This gives the **true performance** of the model.

Important rule:

> The test set should only be checked **once at the end**.

If you keep using the test set to improve the model, it stops being a fair test.

---

# 6. Why the Data Was Split This Way

In the code example, the dataset is split like this:

```
70% Training
15% Validation
15% Test
```

Meaning:

```
100% Total Data

70% → Learn patterns
15% → Practice exam
15% → Final exam
```

---

# Visual Example

```
Full Dataset
     |
     v
-------------------------
|  Train  |  Temp (30%) |
|  70%    |             |
-------------------------
               |
               v
        -----------------
        | Val |  Test  |
        |15%  | 15%    |
        -----------------
```

---

# Key Takeaway

* **Training data** teaches the model.
* **Validation data** helps improve the model.
* **Test data** proves the model works on unseen data.

---

# One Sentence Summary

**Validation data is used to check and improve a model before performing the final test.**
